# MCP with Claude for QGIS Integration & Claude Code Skills

This notebook covers two main topics:
1. **Model Context Protocol (MCP)** - How to use MCP with Claude to work with QGIS
2. **Claude Code Skills** - How to generate custom skills for Claude Code

---

# Part 1: Model Context Protocol (MCP) with Claude for QGIS

## What is MCP?

The **Model Context Protocol (MCP)** is an open standard developed by Anthropic that enables AI assistants like Claude to connect with external data sources and tools. MCP provides a standardized way for Claude to:

- Access local files and databases
- Connect to external APIs and services
- Execute tools in a controlled, secure manner
- Maintain context across different applications

## MCP Architecture

```
+----------------+      +----------------+      +----------------+
|   Claude AI    | <--> |   MCP Server   | <--> |  QGIS / Data   |
|   (Client)     |      |   (Protocol)   |      |   Sources      |
+----------------+      +----------------+      +----------------+
```

### Key Components:
1. **MCP Client**: Claude or any AI that speaks MCP
2. **MCP Server**: The middleware that exposes tools/resources
3. **Resources**: Data sources (files, databases, APIs)

## Setting Up MCP for QGIS Integration

### Step 1: Install Required Packages

In [ ]:
# Install MCP SDK and QGIS-related packages
# !pip install mcp anthropic qgis
# For PyQGIS outside QGIS Desktop:
# !pip install pyqgis

### Step 2: Create an MCP Server for QGIS Operations

Below is a complete example of an MCP server that exposes QGIS tools to Claude.

In [ ]:
# qgis_mcp_server.py - MCP Server for QGIS Integration

import asyncio
import json
from typing import Any, Optional
from pathlib import Path

# MCP imports
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import (
    Tool,
    TextContent,
    Resource,
    ResourceContents,
)

# Geospatial imports
import geopandas as gpd
from shapely.geometry import shape


class QGISMCPServer:
    """
    MCP Server that provides QGIS-like geospatial operations.
    
    This server exposes tools for:
    - Loading vector/raster layers
    - Performing spatial analysis
    - Exporting data to various formats
    """
    
    def __init__(self):
        self.server = Server("qgis-mcp")
        self.layers: dict[str, gpd.GeoDataFrame] = {}
        self._setup_handlers()
    
    def _setup_handlers(self):
        """Register all MCP handlers."""
        
        @self.server.list_tools()
        async def list_tools() -> list[Tool]:
            """Return available QGIS tools."""
            return [
                Tool(
                    name="load_vector_layer",
                    description="Load a vector layer (GeoJSON, Shapefile, GeoPackage) into memory",
                    inputSchema={
                        "type": "object",
                        "properties": {
                            "path": {"type": "string", "description": "Path to the vector file"},
                            "layer_name": {"type": "string", "description": "Name to assign to the layer"},
                        },
                        "required": ["path", "layer_name"]
                    }
                ),
                Tool(
                    name="buffer_analysis",
                    description="Create buffer zones around features",
                    inputSchema={
                        "type": "object",
                        "properties": {
                            "layer_name": {"type": "string", "description": "Name of the layer"},
                            "distance": {"type": "number", "description": "Buffer distance in layer units"},
                            "output_name": {"type": "string", "description": "Name for the output layer"},
                        },
                        "required": ["layer_name", "distance", "output_name"]
                    }
                ),
                Tool(
                    name="spatial_join",
                    description="Perform spatial join between two layers",
                    inputSchema={
                        "type": "object",
                        "properties": {
                            "left_layer": {"type": "string", "description": "Name of the left layer"},
                            "right_layer": {"type": "string", "description": "Name of the right layer"},
                            "predicate": {"type": "string", "description": "Spatial predicate (intersects, within, contains)"},
                            "output_name": {"type": "string", "description": "Name for the output layer"},
                        },
                        "required": ["left_layer", "right_layer", "predicate", "output_name"]
                    }
                ),
                Tool(
                    name="get_layer_info",
                    description="Get information about a loaded layer (CRS, extent, feature count, columns)",
                    inputSchema={
                        "type": "object",
                        "properties": {
                            "layer_name": {"type": "string", "description": "Name of the layer"}
                        },
                        "required": ["layer_name"]
                    }
                ),
                Tool(
                    name="export_layer",
                    description="Export a layer to file (GeoJSON, Shapefile, GeoPackage)",
                    inputSchema={
                        "type": "object",
                        "properties": {
                            "layer_name": {"type": "string", "description": "Name of the layer to export"},
                            "output_path": {"type": "string", "description": "Output file path"},
                            "driver": {"type": "string", "description": "Output format: GeoJSON, ESRI Shapefile, GPKG"}
                        },
                        "required": ["layer_name", "output_path"]
                    }
                ),
                Tool(
                    name="clip_layer",
                    description="Clip one layer by another (like QGIS clip tool)",
                    inputSchema={
                        "type": "object",
                        "properties": {
                            "input_layer": {"type": "string", "description": "Layer to be clipped"},
                            "clip_layer": {"type": "string", "description": "Layer to clip by"},
                            "output_name": {"type": "string", "description": "Name for the output layer"}
                        },
                        "required": ["input_layer", "clip_layer", "output_name"]
                    }
                ),
                Tool(
                    name="calculate_area",
                    description="Calculate area of polygon features",
                    inputSchema={
                        "type": "object",
                        "properties": {
                            "layer_name": {"type": "string", "description": "Name of the polygon layer"},
                            "area_column": {"type": "string", "description": "Name for the new area column"},
                            "unit": {"type": "string", "description": "Area unit: sqm, sqkm, hectares"}
                        },
                        "required": ["layer_name"]
                    }
                ),
            ]
        
        @self.server.call_tool()
        async def call_tool(name: str, arguments: dict) -> list[TextContent]:
            """Execute a QGIS tool."""
            try:
                if name == "load_vector_layer":
                    result = self._load_vector_layer(**arguments)
                elif name == "buffer_analysis":
                    result = self._buffer_analysis(**arguments)
                elif name == "spatial_join":
                    result = self._spatial_join(**arguments)
                elif name == "get_layer_info":
                    result = self._get_layer_info(**arguments)
                elif name == "export_layer":
                    result = self._export_layer(**arguments)
                elif name == "clip_layer":
                    result = self._clip_layer(**arguments)
                elif name == "calculate_area":
                    result = self._calculate_area(**arguments)
                else:
                    result = f"Unknown tool: {name}"
                
                return [TextContent(type="text", text=str(result))]
            except Exception as e:
                return [TextContent(type="text", text=f"Error: {str(e)}")]
    
    def _load_vector_layer(self, path: str, layer_name: str) -> str:
        """Load a vector layer into memory."""
        gdf = gpd.read_file(path)
        self.layers[layer_name] = gdf
        return f"Loaded layer '{layer_name}' with {len(gdf)} features. CRS: {gdf.crs}"
    
    def _buffer_analysis(self, layer_name: str, distance: float, output_name: str) -> str:
        """Create buffer zones around features."""
        if layer_name not in self.layers:
            return f"Layer '{layer_name}' not found"
        
        gdf = self.layers[layer_name].copy()
        gdf['geometry'] = gdf.geometry.buffer(distance)
        self.layers[output_name] = gdf
        return f"Created buffer layer '{output_name}' with distance {distance}"
    
    def _spatial_join(self, left_layer: str, right_layer: str, 
                      predicate: str, output_name: str) -> str:
        """Perform spatial join between two layers."""
        if left_layer not in self.layers or right_layer not in self.layers:
            return "One or both layers not found"
        
        left = self.layers[left_layer]
        right = self.layers[right_layer]
        
        result = gpd.sjoin(left, right, how="inner", predicate=predicate)
        self.layers[output_name] = result
        return f"Spatial join complete. Output '{output_name}' has {len(result)} features"
    
    def _get_layer_info(self, layer_name: str) -> str:
        """Get information about a layer."""
        if layer_name not in self.layers:
            return f"Layer '{layer_name}' not found"
        
        gdf = self.layers[layer_name]
        info = {
            "layer_name": layer_name,
            "crs": str(gdf.crs),
            "feature_count": len(gdf),
            "geometry_type": gdf.geometry.geom_type.unique().tolist(),
            "bounds": gdf.total_bounds.tolist(),
            "columns": gdf.columns.tolist(),
        }
        return json.dumps(info, indent=2)
    
    def _export_layer(self, layer_name: str, output_path: str, 
                      driver: str = "GeoJSON") -> str:
        """Export a layer to file."""
        if layer_name not in self.layers:
            return f"Layer '{layer_name}' not found"
        
        gdf = self.layers[layer_name]
        gdf.to_file(output_path, driver=driver)
        return f"Exported '{layer_name}' to {output_path}"
    
    def _clip_layer(self, input_layer: str, clip_layer: str, output_name: str) -> str:
        """Clip one layer by another."""
        if input_layer not in self.layers or clip_layer not in self.layers:
            return "One or both layers not found"
        
        input_gdf = self.layers[input_layer]
        clip_gdf = self.layers[clip_layer]
        
        result = gpd.clip(input_gdf, clip_gdf)
        self.layers[output_name] = result
        return f"Clipped layer created as '{output_name}' with {len(result)} features"
    
    def _calculate_area(self, layer_name: str, area_column: str = "area", 
                        unit: str = "sqm") -> str:
        """Calculate area of polygon features."""
        if layer_name not in self.layers:
            return f"Layer '{layer_name}' not found"
        
        gdf = self.layers[layer_name]
        
        # Calculate area in square meters first
        if gdf.crs and gdf.crs.is_geographic:
            # Project to equal area CRS for accurate area calculation
            gdf_proj = gdf.to_crs("ESRI:54009")  # World Mollweide
            area_sqm = gdf_proj.geometry.area
        else:
            area_sqm = gdf.geometry.area
        
        # Convert to requested unit
        if unit == "sqkm":
            gdf[area_column] = area_sqm / 1_000_000
        elif unit == "hectares":
            gdf[area_column] = area_sqm / 10_000
        else:
            gdf[area_column] = area_sqm
        
        self.layers[layer_name] = gdf
        return f"Added area column '{area_column}' in {unit} to layer '{layer_name}'"
    
    async def run(self):
        """Run the MCP server."""
        async with stdio_server() as (read_stream, write_stream):
            await self.server.run(
                read_stream,
                write_stream,
                self.server.create_initialization_options()
            )


# Entry point for running the server
if __name__ == "__main__":
    server = QGISMCPServer()
    asyncio.run(server.run())

### Step 3: Configure Claude Code to Use the MCP Server

Create a configuration file for Claude Code to connect to your MCP server.

**Location**: `~/.claude/claude_desktop_config.json` (for Claude Desktop) or project-level `.mcp.json`

In [ ]:
# Example MCP configuration for Claude Code
mcp_config = {
    "mcpServers": {
        "qgis": {
            "command": "python",
            "args": ["qgis_mcp_server.py"],
            "cwd": "/path/to/your/project"
        }
    }
}

# Save this configuration
import json
print(json.dumps(mcp_config, indent=2))

### Step 4: Alternative - Using the MCP Python SDK Directly

You can also use the MCP SDK to create a client that connects to QGIS.

In [ ]:
# MCP Client example - connecting Claude to QGIS tools

from anthropic import Anthropic
import geopandas as gpd
import json


class ClaudeQGISAgent:
    """
    Agent that uses Claude with QGIS-like geospatial tools.
    
    This demonstrates how to give Claude access to geospatial operations
    through function calling (tool use).
    """
    
    def __init__(self, api_key: str = None):
        self.client = Anthropic(api_key=api_key)
        self.layers: dict[str, gpd.GeoDataFrame] = {}
        
        # Define tools that Claude can use
        self.tools = [
            {
                "name": "load_vector_layer",
                "description": "Load a vector layer (GeoJSON, Shapefile, GeoPackage) from a file path or URL",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "path": {
                            "type": "string",
                            "description": "Path or URL to the vector file"
                        },
                        "layer_name": {
                            "type": "string",
                            "description": "Name to assign to the loaded layer"
                        }
                    },
                    "required": ["path", "layer_name"]
                }
            },
            {
                "name": "buffer_analysis",
                "description": "Create buffer zones around features in a layer",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "layer_name": {
                            "type": "string",
                            "description": "Name of the input layer"
                        },
                        "distance": {
                            "type": "number",
                            "description": "Buffer distance in layer units (degrees for EPSG:4326)"
                        },
                        "output_name": {
                            "type": "string",
                            "description": "Name for the output buffer layer"
                        }
                    },
                    "required": ["layer_name", "distance", "output_name"]
                }
            },
            {
                "name": "spatial_join",
                "description": "Join attributes from one layer to another based on spatial relationship",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "left_layer": {
                            "type": "string",
                            "description": "Name of the left/target layer"
                        },
                        "right_layer": {
                            "type": "string",
                            "description": "Name of the right/join layer"
                        },
                        "predicate": {
                            "type": "string",
                            "description": "Spatial predicate: intersects, within, contains",
                            "enum": ["intersects", "within", "contains"]
                        },
                        "output_name": {
                            "type": "string",
                            "description": "Name for the output joined layer"
                        }
                    },
                    "required": ["left_layer", "right_layer", "predicate", "output_name"]
                }
            },
            {
                "name": "get_layer_summary",
                "description": "Get a summary of a layer including CRS, bounds, feature count, and column statistics",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "layer_name": {
                            "type": "string",
                            "description": "Name of the layer to summarize"
                        }
                    },
                    "required": ["layer_name"]
                }
            },
            {
                "name": "export_layer",
                "description": "Export a layer to a file in various formats",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "layer_name": {
                            "type": "string",
                            "description": "Name of the layer to export"
                        },
                        "output_path": {
                            "type": "string",
                            "description": "Output file path"
                        },
                        "format": {
                            "type": "string",
                            "description": "Output format",
                            "enum": ["GeoJSON", "ESRI Shapefile", "GPKG"]
                        }
                    },
                    "required": ["layer_name", "output_path"]
                }
            }
        ]
    
    def execute_tool(self, tool_name: str, tool_input: dict) -> str:
        """Execute a QGIS tool and return the result."""
        
        if tool_name == "load_vector_layer":
            try:
                gdf = gpd.read_file(tool_input["path"])
                self.layers[tool_input["layer_name"]] = gdf
                return f"Successfully loaded layer '{tool_input['layer_name']}' with {len(gdf)} features. CRS: {gdf.crs}"
            except Exception as e:
                return f"Error loading layer: {str(e)}"
        
        elif tool_name == "buffer_analysis":
            layer_name = tool_input["layer_name"]
            if layer_name not in self.layers:
                return f"Layer '{layer_name}' not found"
            
            gdf = self.layers[layer_name].copy()
            gdf['geometry'] = gdf.geometry.buffer(tool_input["distance"])
            self.layers[tool_input["output_name"]] = gdf
            return f"Created buffer layer '{tool_input['output_name']}' with {len(gdf)} features"
        
        elif tool_name == "spatial_join":
            left = self.layers.get(tool_input["left_layer"])
            right = self.layers.get(tool_input["right_layer"])
            
            if left is None or right is None:
                return "One or both layers not found"
            
            result = gpd.sjoin(left, right, how="inner", predicate=tool_input["predicate"])
            self.layers[tool_input["output_name"]] = result
            return f"Spatial join complete. Output has {len(result)} features"
        
        elif tool_name == "get_layer_summary":
            layer_name = tool_input["layer_name"]
            if layer_name not in self.layers:
                return f"Layer '{layer_name}' not found"
            
            gdf = self.layers[layer_name]
            summary = {
                "name": layer_name,
                "crs": str(gdf.crs),
                "feature_count": len(gdf),
                "geometry_types": gdf.geometry.geom_type.unique().tolist(),
                "bounds": gdf.total_bounds.tolist(),
                "columns": list(gdf.columns),
            }
            return json.dumps(summary, indent=2)
        
        elif tool_name == "export_layer":
            layer_name = tool_input["layer_name"]
            if layer_name not in self.layers:
                return f"Layer '{layer_name}' not found"
            
            driver = tool_input.get("format", "GeoJSON")
            self.layers[layer_name].to_file(tool_input["output_path"], driver=driver)
            return f"Exported '{layer_name}' to {tool_input['output_path']}"
        
        return f"Unknown tool: {tool_name}"
    
    def chat(self, user_message: str) -> str:
        """
        Send a message to Claude and handle any tool calls.
        
        Parameters
        ----------
        user_message : str
            The user's request
            
        Returns
        -------
        str
            Claude's response after executing any necessary tools
        """
        messages = [{"role": "user", "content": user_message}]
        
        # Initial request to Claude
        response = self.client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            tools=self.tools,
            messages=messages,
            system="You are a GIS analyst assistant. Use the available tools to help users with geospatial analysis tasks. Always explain what you're doing and why."
        )
        
        # Process tool calls in a loop
        while response.stop_reason == "tool_use":
            # Extract tool use from response
            tool_use_block = next(
                block for block in response.content 
                if block.type == "tool_use"
            )
            
            # Execute the tool
            tool_result = self.execute_tool(
                tool_use_block.name, 
                tool_use_block.input
            )
            
            # Add assistant response and tool result to messages
            messages.append({"role": "assistant", "content": response.content})
            messages.append({
                "role": "user",
                "content": [{
                    "type": "tool_result",
                    "tool_use_id": tool_use_block.id,
                    "content": tool_result
                }]
            })
            
            # Continue the conversation
            response = self.client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=4096,
                tools=self.tools,
                messages=messages,
                system="You are a GIS analyst assistant. Use the available tools to help users with geospatial analysis tasks."
            )
        
        # Extract final text response
        return next(
            block.text for block in response.content 
            if hasattr(block, 'text')
        )

### Example Usage of the Claude QGIS Agent

In [ ]:
# Example usage (requires ANTHROPIC_API_KEY environment variable)
# agent = ClaudeQGISAgent()

# Example queries:
# response = agent.chat("Load the Peru administrative boundaries from GADM and tell me about the layer")
# response = agent.chat("Create a 10km buffer around all the fire points and save as GeoJSON")
# response = agent.chat("Which administrative regions contain the most fire points?")

---

# Part 2: Creating Skills for Claude Code

## What are Claude Code Skills?

**Skills** (also called slash commands) are custom commands that extend Claude Code's capabilities. They allow you to:

- Create reusable workflows
- Define domain-specific operations
- Standardize common tasks
- Share functionality across projects

## Skill Structure

Skills are defined as markdown files with specific frontmatter that tells Claude Code how to execute them.

### Basic Skill Template

```markdown
---
name: skill-name
description: Brief description of what this skill does
---

# Skill Instructions

Your detailed instructions for Claude go here...
```

### Where to Place Skills

Skills can be placed in several locations:

1. **Project-level**: `.claude/skills/` directory in your project
2. **User-level**: `~/.claude/skills/` for global skills
3. **MCP Server**: Skills can be provided by MCP servers

## Example Skills for GeoSpatial Work

In [ ]:
# Let's create the skills directory structure
import os
from pathlib import Path

skills_dir = Path(".claude/skills")
skills_dir.mkdir(parents=True, exist_ok=True)
print(f"Created skills directory: {skills_dir.absolute()}")

### Skill 1: Fire Analysis Skill

In [ ]:
fire_analysis_skill = '''---
name: fire-analysis
description: Analyze fire data for a specific country and year using NASA VIIRS data
---

# Fire Analysis Skill

When the user invokes `/fire-analysis`, perform the following steps:

## Parameters
Parse the user's input for:
- **country**: The country to analyze (default: Peru)
- **year**: The year to fetch data for (required)
- **aggregation**: "year" or "year_month" (default: "year")

## Steps

1. **Load the fire data**:
   - Use the `download_viirs_snpp_peru()` function from `tools.py`
   - Clean the data using `fires_clean()`

2. **Load administrative boundaries**:
   - Use `download_ctry_shp()` to get GADM boundaries

3. **Perform spatial analysis**:
   - Use `intersect_and_collapse_fires_peru()` to aggregate fires by region
   - Include FRP (Fire Radiative Power) sum in aggregations

4. **Generate output**:
   - Create a summary table showing fire counts by region
   - Identify the top 5 regions with most fires
   - Calculate total fires for the period

5. **Optional visualization**:
   - If requested, create a Folium choropleth map using `folium_choropleth_peru()`

## Example Usage
```
/fire-analysis 2023
/fire-analysis Peru 2022 year_month
```

## Output Format
Provide results as a formatted table with:
- Region name
- Fire count
- Total FRP
- Percentage of total fires
'''

# Save the skill
with open(".claude/skills/fire-analysis.md", "w") as f:
    f.write(fire_analysis_skill)
print("Created: .claude/skills/fire-analysis.md")

### Skill 2: GeoJSON Validator Skill

In [ ]:
geojson_validator_skill = '''---
name: validate-geojson
description: Validate and analyze GeoJSON files for correctness and quality
---

# GeoJSON Validator Skill

When the user invokes `/validate-geojson`, perform a comprehensive validation of GeoJSON files.

## Parameters
- **file_path**: Path to the GeoJSON file (required)
- **fix**: If "true", attempt to fix common issues (optional)

## Validation Checks

1. **Structure Validation**:
   - Valid JSON syntax
   - Required GeoJSON properties (type, features/geometry)
   - Feature collection vs single feature

2. **Geometry Validation**:
   - Valid geometry types (Point, LineString, Polygon, etc.)
   - Coordinate validity (within valid ranges)
   - Polygon ring closure
   - No self-intersections

3. **CRS Check**:
   - Check for CRS specification
   - Warn if non-WGS84 coordinates detected

4. **Data Quality**:
   - Null geometries
   - Empty geometries
   - Duplicate features
   - Property consistency across features

## Output Report
Generate a validation report with:
- Summary statistics (feature count, geometry types)
- List of errors found
- List of warnings
- Recommendations for fixing issues

## Example Usage
```
/validate-geojson data/boundaries.geojson
/validate-geojson data/points.geojson fix=true
```
'''

with open(".claude/skills/validate-geojson.md", "w") as f:
    f.write(geojson_validator_skill)
print("Created: .claude/skills/validate-geojson.md")

### Skill 3: Spatial Query Skill

In [ ]:
spatial_query_skill = '''---
name: spatial-query
description: Execute spatial queries on geospatial datasets using natural language
---

# Spatial Query Skill

When the user invokes `/spatial-query`, translate natural language queries into geospatial operations.

## Supported Query Types

### 1. Point-in-Polygon Queries
- "Which region contains point X, Y?"
- "Find all points within polygon Z"

### 2. Distance Queries
- "Find all features within X km of location Y"
- "What is the distance between A and B?"

### 3. Intersection Queries
- "Which features intersect with layer X?"
- "Find overlapping areas between A and B"

### 4. Aggregation Queries
- "Count features by region"
- "Sum attribute X grouped by region Y"

## Implementation Steps

1. **Parse the query** to identify:
   - Operation type (contains, intersects, within, buffer, etc.)
   - Input layers/datasets
   - Parameters (distances, coordinates, attribute names)

2. **Load required data**:
   - Identify and load necessary datasets
   - Ensure CRS compatibility

3. **Execute the operation**:
   - Use GeoPandas spatial operations
   - Handle edge cases and errors gracefully

4. **Return results**:
   - Format output appropriately (table, GeoJSON, summary)
   - Include relevant statistics

## Example Usage
```
/spatial-query Find all fires within 50km of Lima
/spatial-query Count fires by department for 2023
/spatial-query Which regions have more than 1000 fires?
```
'''

with open(".claude/skills/spatial-query.md", "w") as f:
    f.write(spatial_query_skill)
print("Created: .claude/skills/spatial-query.md")

### Skill 4: Map Export Skill

In [ ]:
map_export_skill = '''---
name: export-map
description: Export geospatial data to various map formats (Folium HTML, static image, KML)
---

# Map Export Skill

When the user invokes `/export-map`, create and export maps in various formats.

## Parameters
- **layer**: Name or path of the layer to map (required)
- **format**: Output format - html, png, kml (default: html)
- **style**: Map style - choropleth, points, heatmap (default: auto-detect)
- **output**: Output file path (optional)

## Supported Formats

### 1. Interactive HTML (Folium)
- Creates zoomable, interactive web maps
- Supports multiple tile providers (OSM, CartoDB, Stamen)
- Includes popups and tooltips

### 2. Static PNG/SVG (Matplotlib)
- Publication-quality static maps
- Customizable styling
- Supports north arrows and scale bars

### 3. KML/KMZ (Google Earth)
- Compatible with Google Earth
- Includes styling and descriptions

## Map Components

For choropleth maps:
- Color column selection
- Classification method (quantiles, equal interval, natural breaks)
- Legend generation

For point maps:
- Marker clustering for large datasets
- Custom marker icons
- Popup content configuration

## Example Usage
```
/export-map fires_2023 html
/export-map admin_regions png style=choropleth
/export-map points.geojson kml output=export.kml
```
'''

with open(".claude/skills/export-map.md", "w") as f:
    f.write(map_export_skill)
print("Created: .claude/skills/export-map.md")

### Skill 5: QGIS Project Generator

In [ ]:
qgis_project_skill = '''---
name: create-qgis-project
description: Generate a QGIS project file (.qgz) with layers and styling
---

# QGIS Project Generator Skill

When the user invokes `/create-qgis-project`, generate a QGIS project file with proper layer configuration.

## Parameters
- **name**: Project name (required)
- **layers**: Comma-separated list of layer paths (required)
- **crs**: Project CRS (default: EPSG:4326)
- **output**: Output .qgz file path (optional)

## Features

### Layer Configuration
- Automatic layer ordering (polygons bottom, points top)
- CRS matching and transformation
- Attribute table configuration

### Styling
- Auto-generate styles based on geometry type
- Categorized styling for categorical attributes
- Graduated styling for numeric attributes

### Project Settings
- Set project extent to layer bounds
- Configure default map units
- Add basemap layers (OSM, satellite)

## Implementation Notes

Since QGIS project files are XML-based, this skill will:
1. Create a valid QGIS project XML structure
2. Add layer definitions with proper datasource paths
3. Include basic SLD/QML styling
4. Save as .qgz (compressed) or .qgs (uncompressed)

## Example Usage
```
/create-qgis-project name=Peru_Fires layers=fires.geojson,admin.shp
/create-qgis-project name=Analysis layers=data/*.geojson crs=EPSG:32718
```

## Output
- QGIS project file (.qgz or .qgs)
- Summary of layers added
- Instructions for opening in QGIS
'''

with open(".claude/skills/create-qgis-project.md", "w") as f:
    f.write(qgis_project_skill)
print("Created: .claude/skills/create-qgis-project.md")

## Advanced Skill Features

### Skill with Tool Dependencies

In [ ]:
advanced_skill = '''---
name: geo-pipeline
description: Run a complete geospatial data processing pipeline
allowed_tools:
  - Bash
  - Read
  - Write
  - Glob
---

# Geospatial Pipeline Skill

When the user invokes `/geo-pipeline`, execute a complete data processing workflow.

## Pipeline Stages

### Stage 1: Data Ingestion
- Scan input directory for geospatial files
- Validate file formats and structures
- Log any issues found

### Stage 2: Data Cleaning
- Remove null geometries
- Fix invalid geometries (buffer(0) trick)
- Standardize CRS to EPSG:4326
- Clean attribute names (lowercase, no spaces)

### Stage 3: Processing
- Execute specified spatial operations
- Generate derived attributes
- Perform aggregations

### Stage 4: Export
- Export processed data to specified formats
- Generate metadata files
- Create processing log

## Configuration

The pipeline can be configured via a YAML file:

```yaml
pipeline:
  input_dir: ./data/raw
  output_dir: ./data/processed
  operations:
    - type: buffer
      distance: 1000
    - type: spatial_join
      right_layer: admin_boundaries.geojson
  output_format: GeoJSON
```

## Example Usage
```
/geo-pipeline config=pipeline.yaml
/geo-pipeline input=data/raw output=data/processed
```
'''

with open(".claude/skills/geo-pipeline.md", "w") as f:
    f.write(advanced_skill)
print("Created: .claude/skills/geo-pipeline.md")

## List All Created Skills

In [ ]:
import os
from pathlib import Path

skills_dir = Path(".claude/skills")

print("Created Skills:")
print("=" * 50)

for skill_file in sorted(skills_dir.glob("*.md")):
    # Read the first few lines to get name and description
    with open(skill_file) as f:
        content = f.read()
    
    # Extract name and description from frontmatter
    import re
    name_match = re.search(r'name:\s*(.+)', content)
    desc_match = re.search(r'description:\s*(.+)', content)
    
    name = name_match.group(1) if name_match else skill_file.stem
    desc = desc_match.group(1) if desc_match else "No description"
    
    print(f"\n/{name}")
    print(f"  Description: {desc}")
    print(f"  File: {skill_file}")

---

# Part 3: Putting It All Together

## Complete MCP + Skills Configuration

In [ ]:
# Create the complete project configuration
import json
from pathlib import Path

# Create .claude directory if it doesn't exist
claude_dir = Path(".claude")
claude_dir.mkdir(exist_ok=True)

# MCP configuration
mcp_config = {
    "mcpServers": {
        "qgis-tools": {
            "command": "python",
            "args": ["qgis_mcp_server.py"],
            "env": {
                "PYTHONPATH": "."
            }
        }
    }
}

# Save MCP config
with open(".mcp.json", "w") as f:
    json.dump(mcp_config, f, indent=2)

print("Created .mcp.json")

# Project settings for Claude Code
claude_settings = {
    "permissions": {
        "allow": [
            "Read",
            "Write",
            "Bash(python *)",
            "Bash(pip install *)"
        ]
    },
    "model": "claude-sonnet-4-20250514"
}

with open(".claude/settings.json", "w") as f:
    json.dump(claude_settings, f, indent=2)

print("Created .claude/settings.json")

## Summary

### MCP for QGIS Integration

1. **MCP Server**: Create a Python server that exposes QGIS-like tools
2. **Tool Definitions**: Define tools with clear input schemas
3. **Configuration**: Set up `.mcp.json` to connect Claude Code to your server
4. **Alternative**: Use Claude's tool use API directly with GeoPandas

### Claude Code Skills

1. **Location**: Place skills in `.claude/skills/` directory
2. **Format**: Markdown files with YAML frontmatter
3. **Structure**: Include name, description, and detailed instructions
4. **Usage**: Invoke with `/skill-name` in Claude Code

### Created Skills

| Skill | Command | Description |
|-------|---------|-------------|
| Fire Analysis | `/fire-analysis` | Analyze NASA VIIRS fire data |
| GeoJSON Validator | `/validate-geojson` | Validate GeoJSON files |
| Spatial Query | `/spatial-query` | Natural language spatial queries |
| Map Export | `/export-map` | Export maps to various formats |
| QGIS Project | `/create-qgis-project` | Generate QGIS project files |
| Geo Pipeline | `/geo-pipeline` | Run data processing pipelines |

## References

- [Model Context Protocol Documentation](https://modelcontextprotocol.io/)
- [Claude Code Documentation](https://docs.anthropic.com/claude-code)
- [Anthropic API - Tool Use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- [GeoPandas Documentation](https://geopandas.org/)
- [QGIS Python API](https://qgis.org/pyqgis/)